# 보상 설계가 제어 성능을 어떻게 바꾸는가 — 회전형 역진자 + SAC

앞선 연구 `behavior_cloning_yeen/` 에서 모방학습은 두 가지를 못 했다.

| 남은 문제 | 원인 |
|---|---|
| 세운 막대가 **8.9° 틀어진 채** 유지 | 교사의 숨은 상태 `b_hat` 을 학생이 못 봄 |
| DAgger 31판이 **팔 한계(140°)로 폭주** | 팔 위치를 제약할 수단이 없음 |

모방학습은 **"교사가 뭘 했는지"** 만 말할 수 있다. 강화학습은 **"결과가 어떠해야 하는지"** 를 말한다.
그 말하는 방식 — **보상 함수** — 을 한 항씩 바꿔가며 무엇이 실제로 성능을 만드는지 측정한다.

$$r = \mathrm{up}(\theta) \;-\; k_{arm}\Big(\frac{\phi}{140°}\Big)^2 \;-\; k_{eff}\,a^2
\qquad \text{실패로 종료되면 } +\,\mathrm{term}$$

| 이름 | `up(θ)` | `k_arm` | `k_eff` | `term` | 확인하려는 것 |
|---|---|---|---|---|---|
| `full` | `0.5(1+cos θ)` | 0.1 | 0.01 | −50 | **기준선** |
| `no_arm` | `0.5(1+cos θ)` | **0** | 0.01 | −50 | 팔이 폭주하는가 |
| `no_effort` | `0.5(1+cos θ)` | 0.1 | **0** | −50 | 명령만 커지는가 |
| `sparse` | **±10° 안 1, 밖 0** | 0.1 | 0.01 | −50 | 학습이 시작되는가 |
| `trap` | **−θ²** | 0 | 0 | **0** | **종료가 이득** — 일부러 실패시킨 설계 |

**실행 순서:** 셀을 위에서부터 차례로 돌리면 된다. 전체 **약 30분**.

## 1. 저장소 받기

내 코드는 `Yeeun/rl_yeen/`, 플랜트 모델은 교수님의 `simulator/rotary_model.py` 를 쓴다.
`os.path.exists` 로 감싸서 **여러 번 돌려도 안전**하다.

In [ ]:
import os
if not os.path.exists("/content/Pendulum_control"):
    !git clone -q https://github.com/jooyongsim/Pendulum_control.git
os.chdir("/content/Pendulum_control")
os.environ["PYTHONPATH"] = "Yeeun/rl_yeen"

import sys
sys.path.insert(0, "/content/Pendulum_control/Yeeun/rl_yeen")

import torch
print("작업 폴더:", os.getcwd())
print("torch", torch.__version__, "| GPU는 쓰지 않는다 — 신경망이 작고 물리 적분이 병목")

## 2. 환경과 보상 먼저 확인하기

학습을 돌리기 전에 **환경이 제대로 동작하는지**, **보상이 의도대로 생겼는지** 본다.

- 입력 없이 직립 5° 에서 놓으면 넘어져서 매달린 쪽으로 가야 한다 (역진자는 불안정).
- `teacher` 는 실기의 에너지 스윙업 + 극점 배치 제어기다. **RL이 넘어야 할 선.**
- 마지막 표가 `trap` 의 함정을 **숫자로** 보여준다.

In [ ]:
import math, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import rl_common as rl
import eval_rl as ev

# --- 2-1. 환경 점검: 무입력, 직립 5도에서 출발 ---
lg = rl.rollout(lambda o: 0.0, "swingup", "full", theta0=math.radians(5), seconds=3.0)
plt.figure(figsize=(7, 2.4))
plt.plot(lg["t"], np.degrees(lg["th"]))
plt.axhspan(-10, 10, color="#2E8B57", alpha=.15)
plt.xlabel("t [s]"); plt.ylabel("theta from up [deg]")
plt.title("free fall from 5 deg (u = 0)"); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

# --- 2-2. 기준선: 실기 제어기 ---
print("teacher (RL이 넘어야 할 선)")
for task in ("balance", "swingup"):
    m = rl.evaluate(ev.teacher, task, "full")
    print(f"  {task:<8s} return {m['ret']:7.1f}  hold10 {m['hold10']:5.1f}%  hold2 {m['hold2']:5.1f}%"
          f"  bias {m['bias_deg']:+.2f} deg  arm {m['arm_max_deg']:.1f} deg")

# --- 2-3. trap 의 함정을 숫자로 ---
rows = []
for name, cfg in rl.REWARDS.items():
    fall = rl.rollout(lambda o: 0.0, "balance", name, theta0=0.1)      # 아무것도 안 하고 넘어짐
    hold = rl.rollout(ev.teacher,   "balance", name, theta0=0.1)       # 끝까지 버팀
    rows.append({"보상": name, "설명": cfg["doc"],
                 "넘어짐 (17스텝)": fall["r"].sum(), "버팀 (250스텝)": hold["r"].sum()})
df = pd.DataFrame(rows).set_index("보상")
df["버티는 게 이득?"] = np.where(df["버팀 (250스텝)"] > df["넘어짐 (17스텝)"], "O", "X  ← 함정")
display(df.style.format({"넘어짐 (17스텝)": "{:9.2f}", "버팀 (250스텝)": "{:9.2f}"})
    .set_caption("보상이 항상 음수고 실패 벌점이 없으면, 빨리 끝내는 쪽이 리턴이 크다")
    .set_properties(**{"text-align": "right", "padding": "6px 12px"})
    .set_table_styles([{"selector": "caption",
                        "props": "caption-side:bottom;font-size:0.82em;color:#667;padding-top:10px;text-align:left;"}]))

## 3. 보상 5개 학습

`train_sac.py --sweep` 이 `rl_common.REWARDS` 의 다섯 가지를 차례로 학습한다.
과제는 **balance** (직립 근처 시작, 5초) — swing-up 보다 훨씬 빨라서 비교에 적합하다.

진행 중 출력에서 볼 것:

```
   step/steps   train ...   eval ...   hold10 ...   bias ...   arm ...   fail n/4   alpha ...
```

- **`bias`** — 중심이 0에서 틀어진 정도. 모방학습에서 −8.9° 였던 그 값.
- **`arm`** — 팔 최대 각도. `no_arm` 이 여기서 커지는지 보라.
- **`alpha`** — 탐색 정도를 스스로 조절하는 값. 학습이 진행되면 줄어든다.

> ⏱ 5개 × 30,000 스텝 = **약 20~25분**.

In [ ]:
!python Yeeun/rl_yeen/train_sac.py --sweep --task balance --steps 30000 --seed 0

## 4. 같은 잣대로 채점하고 눈으로 보기

**자기가 학습한 보상으로 잰 리턴은 측정이 아니라 자기 평가다.**
`trap` 은 짧은 에피소드에 `−θ²` 를 적용해 "우승"해버린다.
그래서 모두 `full` 로 다시 채점한다.

그리고 직립 판정을 **±10° / ±5° / ±2°** 세 가지로 본다 —
모방학습 모델이 **8.9° 틀어진 채 ±10° 를 100% 통과**했기 때문이다.

In [ ]:
import math, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Wedge
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import rl_common as rl, eval_rl as ev

TASK, BAND = "balance", 10.0
PAL = {"teacher": "#4B5563", "full": "#2E8B57", "no_arm": "#E07A3F",
       "no_effort": "#3E7CB1", "sparse": "#8C6D31", "trap": "#C0392B"}

agents = ev.agents(TASK, rl.CKPT_DIR, include_teacher=True)
runs, rows = [], []
for name, act, rep in agents:
    lg = rl.rollout(act, TASK, "full", theta0=math.radians(11))   # 같은 초기조건으로 한 판
    m = rl.metrics(lg)
    e = rl.evaluate(act, TASK, "full")                            # 4개 시작점 평균
    runs.append(dict(name=name, color=PAL.get(name, "#777"), lg=lg))
    rows.append({"에이전트": name, "학습 보상": (rep or {}).get("reward", "— (실기 제어기)"),
                 "return": e["ret"], "직립 ±10°": e["hold10"], "직립 ±5°": e["hold5"],
                 "직립 ±2°": e["hold2"], "편차 [deg]": e["bias_deg"], "떨림 [deg]": e["wobble_deg"],
                 "팔 최대 [deg]": e["arm_max_deg"], "u_rms": e["u_rms"],
                 "실패": f"{int(e['failures'])}/4" + (f" ({e['why']})" if e["why"] != "-" else "")})

# ---------------- 시뮬레이션: 전부 나란히 ----------------
n = len(runs); ncol = 3; nrow = -(-n // ncol)
L, lim = rl.rm.L, 1.34 * rl.rm.L
fig = plt.figure(figsize=(3.6 * ncol, 2.9 * nrow + 2.6), dpi=68)
ph = 0.80 / nrow
panels = []
for i, r in enumerate(runs):
    row, col = i // ncol, i % ncol
    ax = fig.add_axes([0.03 + col * (0.94 / ncol), 0.97 - ph * (row + 1) - 0.30 * row / nrow,
                       0.94 / ncol * 0.90, ph * 0.86])
    ax.set_aspect("equal"); ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xticks([]); ax.set_yticks([])
    ax.add_patch(Wedge((0, 0), L * 1.06, 90 - BAND, 90 + BAND, facecolor="#2E8B57",
                       alpha=.16, edgecolor="none"))
    ax.add_patch(plt.Circle((0, 0), L, fill=False, color="0.9", ls="--", lw=.7))
    rod, = ax.plot([], [], lw=6, color=r["color"], solid_capstyle="round")
    bob, = ax.plot([], [], "o", ms=8, color=r["color"])
    ax.plot([0], [0], "ko", ms=4)
    ax.set_title(r["name"], fontsize=11, weight="bold", color=r["color"])
    txt = ax.text(0, -1.26 * L, "", ha="center", fontsize=9, family="monospace")
    panels.append((rod, bob, txt))

ax_t = fig.add_axes([0.07, 0.05, 0.89, 0.17])
lines, heads = [], []
for r in runs:
    ln, = ax_t.plot([], [], lw=1.6, color=r["color"], label=r["name"])
    hd, = ax_t.plot([], [], "o", ms=4, color=r["color"])
    lines.append(ln); heads.append(hd)
ax_t.axhspan(-BAND, BAND, color="#2E8B57", alpha=.16)
ax_t.set_xlim(0, max(r["lg"]["t"][-1] for r in runs)); ax_t.set_ylim(-45, 45)
ax_t.set_xlabel("t [s]", fontsize=9); ax_t.set_ylabel("theta [deg]", fontsize=9)
ax_t.tick_params(labelsize=8); ax_t.grid(alpha=.3); ax_t.legend(ncol=n, fontsize=8, loc="lower right")

nmax = max(len(r["lg"]["t"]) for r in runs)
idx = np.arange(0, nmax, 2)

def frame(k):
    out = []
    for (rod, bob, txt), ln, hd, r in zip(panels, lines, heads, runs):
        T, TH = r["lg"]["t"], np.degrees(r["lg"]["th"])
        i = min(idx[k], len(T) - 1)
        th = math.radians(TH[i]); x, y = L * math.sin(th), L * math.cos(th)
        ins = abs(TH[i]) < BAND
        rod.set_data([0, x], [0, y]); bob.set_data([x], [y])
        bob.set_color("#2E8B57" if ins else r["color"])
        dead = idx[k] >= len(T)
        txt.set_text("종료" if dead else f"{TH[i]:+6.1f} deg")
        txt.set_color("#C0392B" if dead else ("#2E8B57" if ins else "0.45"))
        ln.set_data(T[:i + 1], TH[:i + 1]); hd.set_data([T[i]], [TH[i]])
        out += [rod, bob, txt, ln, hd]
    return out

ani = FuncAnimation(fig, frame, frames=len(idx), interval=40, blit=True)
plt.close(fig)
display(HTML(ani.to_jshtml()))

# ---------------- 표 ----------------
def grade(s):
    return ["color:#1C6B45;font-weight:700" if v >= 95 else
            ("color:#B8860B;font-weight:700" if v >= 60 else "color:#C0392B;font-weight:700") for v in s]

df = pd.DataFrame(rows).set_index("에이전트")
display(df.style
    .format({"return": "{:.1f}", "직립 ±10°": "{:.0f}%", "직립 ±5°": "{:.0f}%", "직립 ±2°": "{:.0f}%",
             "편차 [deg]": "{:+.2f}", "떨림 [deg]": "{:.2f}", "팔 최대 [deg]": "{:.1f}", "u_rms": "{:.2f}"})
    .apply(grade, subset=["직립 ±10°", "직립 ±5°", "직립 ±2°"])
    .background_gradient(cmap="Oranges", subset=["팔 최대 [deg]"], vmin=0, vmax=140)
    .background_gradient(cmap="Reds", subset=["편차 [deg]"], vmin=-10, vmax=10)
    .set_caption("모두 full 보상으로 채점 · 4개 시작점 평균 · teacher 는 실기 제어기")
    .set_properties(**{"text-align": "right", "padding": "6px 12px"})
    .set_table_styles([{"selector": "caption",
                        "props": "caption-side:bottom;font-size:0.82em;color:#667;padding-top:10px;text-align:left;"},
                       {"selector": "th", "props": "text-align:right;padding:6px 12px;"},
                       {"selector": "tbody tr:nth-child(odd)", "props": "background-color:#ffffff;"}]))

## 5. 모델 오차에 견디는가 (sim-to-real)

식별된 플랜트 상수는 정확하지 않다. 실기에 올리면 `b` 가 다르고 마찰이 더 크다.
학습 때 본 적 없는 조건에서도 버티는지 본다.

- `b ±20%` — 팔이 막대에 주는 영향의 세기
- `2x fric` — 건마찰 2배

In [ ]:
!python Yeeun/rl_yeen/eval_rl.py --task balance --score full --suite robust

## 6. 다음 — swing-up

balance 에서 가장 좋았던 보상으로 **매달린 상태에서 세우기**를 학습한다.
스텝 수가 훨씬 많다 (200,000 ≈ 67분 분량의 경험, **CPU 로 20~40분**).

학습 중 **"빙빙 돌리기" 국소 최적해** 를 보게 될 수 있다 —
세우지 않고 계속 돌리기만 해도 `0.5(1+cos θ)` 의 평균이 꽤 높기 때문이다.
보상 설계가 어려운 이유가 여기 있다.

> 시간이 없으면 이 셀은 건너뛰고, 마지막 저장 셀만 돌린다.

In [ ]:
!python Yeeun/rl_yeen/train_sac.py --task swingup --reward full --steps 200000 --eval-every 10000

In [ ]:
# swing-up 결과 보기 (위 셀을 돌렸을 때만)
import math, numpy as np
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import matplotlib.pyplot as plt
import rl_common as rl, eval_rl as ev

p = rl.CKPT_DIR / "sac_swingup_full.npz"
if not p.exists():
    print("아직 없음 — 위 셀을 먼저 돌리세요")
else:
    pol = rl.NumpyActor(p)
    print("teacher :", {k: round(v, 2) for k, v in rl.evaluate(ev.teacher, "swingup", "full").items()
                        if k in ("ret", "hold10", "hold2", "bias_deg", "arm_max_deg")})
    print("SAC     :", {k: round(v, 2) for k, v in rl.evaluate(pol, "swingup", "full").items()
                        if k in ("ret", "hold10", "hold2", "bias_deg", "arm_max_deg")})
    lg = rl.rollout(pol, "swingup", "full", theta0=math.pi)
    L = rl.rm.L
    fig, ax = plt.subplots(figsize=(4.2, 4.2), dpi=72)
    ax.set_aspect("equal"); ax.set_xlim(-1.3*L, 1.3*L); ax.set_ylim(-1.3*L, 1.3*L)
    ax.set_xticks([]); ax.set_yticks([])
    ax.add_patch(plt.Circle((0, 0), L, fill=False, color="0.88", ls="--", lw=.8))
    rod, = ax.plot([], [], lw=7, color="#2E8B57", solid_capstyle="round")
    ax.plot([0], [0], "ko", ms=5)
    txt = ax.text(0, -1.22*L, "", ha="center", family="monospace", fontsize=9)
    TH, PH, T = np.degrees(lg["th"]), np.degrees(lg["phi"]), lg["t"]
    def f(i):
        th = math.radians(TH[i])
        rod.set_data([0, L*math.sin(th)], [0, L*math.cos(th)])
        rod.set_color("#2E8B57" if abs(TH[i]) < 10 else "#E07A3F")
        txt.set_text(f"t={T[i]:5.2f}s  theta={TH[i]:+6.1f}  arm={PH[i]:+6.1f}")
        return rod, txt
    display(HTML(FuncAnimation(fig, f, frames=range(0, len(T), 2), interval=40, blit=True).to_jshtml()))
    plt.close(fig)

## 7. 결과 저장

Colab 세션이 끊기면 `/content` 가 비워진다. **학습한 모델을 꼭 받아둘 것.**

In [ ]:
!cd /content/Pendulum_control && zip -qr /content/rl_yeen_checkpoints.zip Yeeun/rl_yeen/checkpoints
from google.colab import files
files.download("/content/rl_yeen_checkpoints.zip")